In [143]:
import pandas as pd
import numpy as np

# Read the excel file

In [144]:
df=pd.read_excel('Input.xlsx')
df.head()

,Fiscal Quarter ID,Fiscal Period ID,Corporate Bookings Flag,Organization Code,Bookings Adjustments Type,Bookings Adjustments Code,Delivery Option Code,Monetization,Product Category,Product Classification,Software Type,Revenue Recognition Flag,Accounting Rule,Product Family,Txn Net Price Indicator,XCAT Flag,Business Model,Org Category,Product Bookings Net
0,2025Q2,202504,Y,-,BOOKING,RB,UNKNOWN,Perpetual,Software,Software,On-Premise Perpetual,Y,NaN,HCIPPSW,=,N,Physical Delivery,Adj. Org,-80.06
1,2025Q2,202504,Y,-,BOOKING,RB,UNKNOWN,Term,Hardware,Hardware,On-Premise Subscription,Y,NaN,UCSHX,=,N,Physical Delivery,Adj. Org,-10774.67
2,2025Q2,202504,Y,-,BOOKING,RB,UNKNOWN,Term,Software,Software,On-Premise Subscription,Y,NaN,HCIPS,=,N,Non Factory,Adj. Org,-754.32
3,2025Q2,202504,Y,-,BOOKING,RB,UNKNOWN,Term,Software,Software,On-Premise Subscription,Y,NaN,HCISW,=,N,Non Factory,Adj. Org,-9439.98
4,2025Q2,202504,Y,-,BOOKING,RB,UNKNOWN,Term,Software,Software,On-Premise Subscription,Y,NaN,HXDP,=,N,Non Factory,Adj. Org,-38.30


# things to do:
First check the unique values of index;

Second check the NaN values of columns;

Data cleaning- NaN and index issue;

Filter and get the pivots;

Check with the excel output with the script output;

Import to excel;

# Check the datatype of all the columns

In [145]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147359 entries, 0 to 147358
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Fiscal Quarter ID          147359 non-null  object 
 1   Fiscal Period ID           147359 non-null  int64  
 2   Corporate Bookings Flag    147359 non-null  object 
 3   Organization Code          147359 non-null  object 
 4   Bookings Adjustments Type  97553 non-null   object 
 5   Bookings Adjustments Code  147359 non-null  object 
 6   Delivery Option Code       147359 non-null  object 
 7   Monetization               147359 non-null  object 
 8   Product Category           147359 non-null  object 
 9   Product Classification     147359 non-null  object 
 10  Software Type              147359 non-null  object 
 11  Revenue Recognition Flag   147359 non-null  object 
 12  Accounting Rule            123455 non-null  object 
 13  Product Family             14

# Check if any column has NaN value

In [146]:
df.isna().sum()

Fiscal Quarter ID                0
Fiscal Period ID                 0
Corporate Bookings Flag          0
Organization Code                0
Bookings Adjustments Type    49806
Bookings Adjustments Code        0
Delivery Option Code             0
Monetization                     0
Product Category                 0
Product Classification           0
Software Type                    0
Revenue Recognition Flag         0
Accounting Rule              23904
Product Family                   0
Txn Net Price Indicator          0
XCAT Flag                        0
Business Model                   0
Org Category                     0
Product Bookings Net             0
dtype: int64

# Checked the NaN column-Bookings Adjustments Type will be used for business purpose, so NaN will be replaced by 0

In [124]:
df['Bookings Adjustments Type']

0              BOOKING
1              BOOKING
2              BOOKING
3              BOOKING
4              BOOKING
              ...     
147354             NaN
147355             NaN
147356             NaN
147357    DSV Bookings
147358    DSV Bookings
Name: Bookings Adjustments Type, Length: 147359, dtype: object

# to avoid warnings in the code

In [147]:
import warnings
warnings.filterwarnings('ignore')

# Data Cleaning-part 1
# Convert NaN to 0 for the specific column

In [148]:
df['Bookings Adjustments Type'] = df['Bookings Adjustments Type'].fillna(0)
df['Bookings Adjustments Type']

0              BOOKING
1              BOOKING
2              BOOKING
3              BOOKING
4              BOOKING
              ...     
147354               0
147355               0
147356               0
147357    DSV Bookings
147358    DSV Bookings
Name: Bookings Adjustments Type, Length: 147359, dtype: object

# Data cleaning-part 2
# NON FACTORY and non factory issue solving with Lambda function:

In [150]:
df['Business Model'].unique()

array(['Physical Delivery', 'Non Factory', 'XCAT-MSJ', 'Traditional MSJ',
       'Electronic', 'NON FACTORY', 'Acquisition'], dtype=object)

In [152]:
df['Business Model']=df['Business Model'].apply(lambda x: str.lower(x))
df['Business Model'].unique()

array(['physical delivery', 'non factory', 'xcat-msj', 'traditional msj',
       'electronic', 'acquisition'], dtype=object)

# Standard Net Bookings

In [153]:
filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type']=='BOOKING') |
                (df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type']==0)]
                 
p4 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p4

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
acquisition,3.977391e+08
electronic,1.483513e+07
non factory,5.054717e+08
physical delivery,8.726054e+08
traditional msj,7.331381e+07
xcat-msj,-1.739578e+06


# 2 tier DSV LPOS WPL

In [90]:
df['Txn Net Price Indicator'].unique()

array(['=', 'Y', 'N'], dtype=object)

In [154]:
c1=['DSV Bookings',0] # list created with conditions for df['Bookings Adjustments Type']

filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type'].isin(c1)) & (df['Txn Net Price Indicator']=='N')] 

p5 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p5

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
non factory,13129.363815
physical delivery,12013.800000


# FILTER-2tierposnet

In [155]:
c1=['DSV Bookings',0] # list created with conditions for df['Bookings Adjustments Type']

filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type'].isin(c1)) & (df['Txn Net Price Indicator']=='Y')] 

p6 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p6

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,5.281116e+06
non factory,2.416524e+08
physical delivery,3.069339e+08
traditional msj,3.905091e+07


# FILTER-stdfctrnet

In [156]:

filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type']=='BOOKING')]
p7 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p7

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
acquisition,3.977391e+08
electronic,-3.636784e+05
non factory,-5.941701e+07
physical delivery,-1.246890e+08
traditional msj,-5.633781e+06
xcat-msj,-6.487917e+07


# FILTER-stdfacgrosbkg

In [157]:
filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type']==0)]
p8 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p8

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,1.519881e+07
non factory,5.648888e+08
physical delivery,9.972944e+08
traditional msj,7.894759e+07
xcat-msj,6.313960e+07


# FILTER-stdnonfacgros

In [158]:
c1=['BOOKING',0] # list created with conditions for df['Bookings Adjustments Type']

filtered_df = df[(df['Corporate Bookings Flag'] == 'Y') & (df['Bookings Adjustments Type'].isin(c1))] 

p9 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p9

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
acquisition,3.977391e+08
electronic,1.483513e+07
non factory,5.054717e+08
physical delivery,8.726054e+08
traditional msj,7.331381e+07
xcat-msj,-1.739578e+06


# FILTER-2tierfactorynetbookingwpl

In [159]:
c1=['BOOKING',0] # list created with conditions for df['Bookings Adjustments Type']
c2=['=','N'] # List created with conditions for df['Txn Net Price Indicator']

filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type'].isin(c1)) & (df['Txn Net Price Indicator'].isin(c2))] 

p10 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p10

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,5.235159e+05
non factory,1.746822e+06
physical delivery,1.626274e+08
traditional msj,-6.191227e+04


# FILTER-2-Tier Factory Net (net price)

In [160]:
c1=['BOOKING',0] # list created with conditions for df['Bookings Adjustments Type']


filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type'].isin(c1)) & (df['Txn Net Price Indicator']=='Y')] 

p11 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p11

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,6.731241e+06
non factory,1.614146e+08
physical delivery,4.238811e+08
traditional msj,5.271730e+07


# FILTER-2-Tier Factory, NTG (WPL)

In [161]:
c2=['=','N'] # List created with conditions for df['Txn Net Price Indicator']


filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type']=='BOOKING') & (df['Txn Net Price Indicator'].isin(c2))] 

p12 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p12

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,-2.914500e+02
non factory,-3.602419e+06
physical delivery,-1.708401e+08
traditional msj,-7.338030e+04


# FILTER- 2-Tier Factory NTG Bookingsnet

In [162]:
filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type']=='BOOKING') & (df['Txn Net Price Indicator']=='Y')] 

p13 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p13

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,-6.265643e+04
non factory,-8.039666e+05
physical delivery,-9.345785e+06
traditional msj,-1.123325e+06


# FILTER- 2-Tier Factory, Gross (WPL)

In [163]:
c2=['=','N'] # List created with conditions for df['Txn Net Price Indicator']


filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type']==0) & (df['Txn Net Price Indicator'].isin(c2))] 

p14 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p14

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,5.238074e+05
non factory,5.349241e+06
physical delivery,3.334675e+08
traditional msj,1.146803e+04


# FILTER-2-Tier NonFactory, Gross (WPL)

In [164]:
c1=['BOOKING',0] # list created with conditions for df['Bookings Adjustments Type']
c2=['=','N'] # List created with conditions for df['Txn Net Price Indicator']

filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type'].isin(c1)) & (df['Txn Net Price Indicator'].isin(c2))] 

p15 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p15

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,5.235159e+05
non factory,1.746822e+06
physical delivery,1.626274e+08
traditional msj,-6.191227e+04


# FILTER-2-Tier Factory & Non-Factory Gross (net price)

In [165]:
filtered_df = df[(df['Corporate Bookings Flag'] == 'N') & (df['Bookings Adjustments Type']==0) & (df['Txn Net Price Indicator']=='Y')] 

p16 = pd.pivot_table(filtered_df, 
                    columns=['Fiscal Quarter ID', 'Fiscal Period ID'], 
                    index='Business Model', 
                    values='Product Bookings Net', 
                    aggfunc=sum)
p16

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
electronic,6.793897e+06
non factory,1.622185e+08
physical delivery,4.332269e+08
traditional msj,5.384063e+07


# Filter index using loc function:

In [167]:
p16.loc[['physical delivery','traditional msj']] # pass 2D array for dataframe

Fiscal Quarter ID,2025Q2
Fiscal Period ID,202504
Business Model,
physical delivery,4.332269e+08
traditional msj,5.384063e+07


# Import pivot to Excel with the values in integer